In [2]:
import pandas as pd

mycsv = 'gold_training_data_v2.csv'
df = pd.read_csv(mycsv)
print(f"{mycsv} successfully read")
print(df.head())


gold_training_data_v2.csv successfully read
                  time      open      high       low     close  tick_volume  \
0  2023-07-27 07:00:00  1974.774  1975.613  1974.223  1975.460          452   
1  2023-07-27 07:05:00  1975.444  1975.966  1974.809  1975.902          372   
2  2023-07-27 07:10:00  1975.868  1978.472  1975.644  1978.252          423   
3  2023-07-27 07:15:00  1978.287  1979.935  1978.052  1979.086          543   
4  2023-07-27 07:20:00  1979.080  1979.581  1978.620  1979.098          424   

   spread  real_volume    RSI  RSI_Lag1  ...  Hour  DayOfWeek     ROC_1  \
0     200            0    0.0       0.0  ...     7          3  0.000000   
1     200            0  100.0       0.0  ...     7          3  0.022375   
2     200            0  100.0     100.0  ...     7          3  0.118933   
3     200            0  100.0     100.0  ...     7          3  0.042158   
4     200            0  100.0     100.0  ...     7          3  0.000606   

      ROC_3  ROC_5  Rel_Body  

In [3]:
#convert to date time format
df['time'] = pd.to_datetime(df['time'])


In [4]:
sum_null= df.isnull().sum()
print(sum_null)

time              0
open              0
high              0
low               0
close             0
tick_volume       0
spread            0
real_volume       0
RSI               0
RSI_Lag1          0
RSI_Lag2          0
EMA_50            0
EMA_200           0
Dist_EMA_50       0
Dist_EMA_200      0
ADX               0
ATR               0
Hour              0
DayOfWeek         0
ROC_1             0
ROC_3             0
ROC_5             0
Rel_Body          0
Rel_Upper_Wick    0
Rel_Lower_Wick    0
Future_Close      0
Target            0
dtype: int64


In [5]:
# 1. Check target distribution (class balance)
print(df["Target"].value_counts(normalize=True))

# 2. View all available columns
print(df.columns.tolist())

Target
1    0.51468
0    0.48532
Name: proportion, dtype: float64
['time', 'open', 'high', 'low', 'close', 'tick_volume', 'spread', 'real_volume', 'RSI', 'RSI_Lag1', 'RSI_Lag2', 'EMA_50', 'EMA_200', 'Dist_EMA_50', 'Dist_EMA_200', 'ADX', 'ATR', 'Hour', 'DayOfWeek', 'ROC_1', 'ROC_3', 'ROC_5', 'Rel_Body', 'Rel_Upper_Wick', 'Rel_Lower_Wick', 'Future_Close', 'Target']


In [6]:
bins = [0.0,0.25,0.50,1.0]
labels = ["Small (<25%)", "Medium (25-50%)", "Dominant (>50%)"]
df["Lower_Wick_Bucket"] = pd.cut(df["Rel_Lower_Wick"], labels=labels, bins=bins, include_lowest=True)
df["Upper_Wick_Bucket"] = pd.cut(df["Rel_Upper_Wick"], labels=labels, bins=bins,include_lowest=True)

lower_wick_analysis = df.groupby("Lower_Wick_Bucket", observed=False).agg(
    total_candles = ("Target","count"),
    up_move_prob=("Target", "mean")

).reset_index()
lower_wick_analysis["up_move_prob"] = (lower_wick_analysis["up_move_prob"] * 100).round(2).astype(str) + "%"

upper_wick_analysis = df.groupby("Upper_Wick_Bucket",observed=False).agg(
    total_candles = ("Target","count"),
    up_move_prob = ('Target','mean')
).reset_index()
upper_wick_analysis["up_move_prob"] = (upper_wick_analysis["up_move_prob"] * 100).round(2).astype(str) + "%"




In [7]:
print("--- Lower Wick Rejection (Expecting UP / Target = 1) ---")
print(lower_wick_analysis)

print("\n--- Upper Wick Rejection (Expecting DOWN / Target = 0) ---")
print(upper_wick_analysis)

--- Lower Wick Rejection (Expecting UP / Target = 1) ---
  Lower_Wick_Bucket  total_candles up_move_prob
0      Small (<25%)         104625       51.59%
1   Medium (25-50%)          61973       51.49%
2   Dominant (>50%)          33398       51.06%

--- Upper Wick Rejection (Expecting DOWN / Target = 0) ---
  Upper_Wick_Bucket  total_candles up_move_prob
0      Small (<25%)         108392       51.26%
1   Medium (25-50%)          60539       51.59%
2   Dominant (>50%)          31065       51.97%


In [9]:
df['is_uptrend']= df['close'] > df['EMA_200']
bullish_trend_wicks = df[
    (df['Lower_Wick_Bucket'] == "Dominant (>50%)" ) & 
    (df["is_uptrend"] == True)

]

bullish_win_rate = bullish_trend_wicks["Target"].mean() * 100
bullish_count = len(bullish_trend_wicks)

# 3. Bearish Alignment: Dominant Upper Wick (>50%) during a Downtrend
bearish_trend_wicks = df[
    (df["Upper_Wick_Bucket"] == "Dominant (>50%)") & 
    (df["is_uptrend"] == False)
]

# For bearish trades, Target == 0 is a win (price drops)
bearish_win_rate = (bearish_trend_wicks["Target"] == 0).mean() * 100
bearish_count = len(bearish_trend_wicks)

# Print comparison vs baseline
baseline_up = df["Target"].mean() * 100
baseline_down = (1 - df["Target"].mean()) * 100

print(f"--- Bullish Trend Rejection (Uptrend + Long Lower Wick) ---")
print(f"Total Candles: {bullish_count:,}")
print(f"UP Win Rate:   {bullish_win_rate:.2f}%  (Baseline: {baseline_up:.2f}%)")

print(f"\n--- Bearish Trend Rejection (Downtrend + Long Upper Wick) ---")
print(f"Total Candles: {bearish_count:,}")
print(f"DOWN Win Rate: {bearish_win_rate:.2f}%  (Baseline: {baseline_down:.2f}%)")

--- Bullish Trend Rejection (Uptrend + Long Lower Wick) ---
Total Candles: 18,323
UP Win Rate:   50.32%  (Baseline: 51.47%)

--- Bearish Trend Rejection (Downtrend + Long Upper Wick) ---
Total Candles: 13,196
DOWN Win Rate: 48.21%  (Baseline: 48.53%)


In [10]:
# Calculate correlation of all numeric features with Target
numeric_df = df.select_dtypes(include=['float64', 'int64'])
correlations = numeric_df.corr()["Target"].sort_values(ascending=False)

print("Correlation of Features with Target:")
print(correlations)

Correlation of Features with Target:
Target            1.000000
tick_volume       0.009068
ADX               0.007661
Future_Close      0.007154
Hour              0.006974
Rel_Upper_Wick    0.005885
EMA_200           0.004397
EMA_50            0.004271
open              0.004129
high              0.004112
low               0.004102
close             0.004084
ATR               0.002822
Rel_Body         -0.002429
Rel_Lower_Wick   -0.002806
DayOfWeek        -0.004417
spread           -0.006461
Dist_EMA_200     -0.006803
RSI_Lag2         -0.012010
RSI_Lag1         -0.016571
ROC_1            -0.016660
Dist_EMA_50      -0.017458
ROC_5            -0.019395
ROC_3            -0.020524
RSI              -0.023173
real_volume            NaN
Name: Target, dtype: float64


In [12]:
Extreme_Oversold = df[
    df['RSI'] < 20
    ]

Extreme_Oversold_value = Extreme_Oversold['Target'].mean()*100

Extreme_Overbought = df[
    df['RSI'] > 80
]
Extreme_Overbought_value =( Extreme_Overbought['Target'] == 0).mean()*100
print(f'Extreme_Oversold_value = {Extreme_Oversold_value:.2f} %')
print(f'Extreme_Overvought_value = {Extreme_Overbought_value:.2f} %')

Extreme_Oversold_value = 55.74 %
Extreme_Overvought_value = 48.24 %


In [13]:
# 1. Chronological Train/Test Split (70% Train, 30% Test)
split_idx = int(len(df) * 0.70)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

# 2. In-Sample Check (Train Set)
train_oversold = train_df[train_df["RSI"] < 20]
train_win_rate = train_oversold["Target"].mean() * 100

# 3. Out-of-Sample Check (Unseen Test Set)
test_oversold = test_df[test_df["RSI"] < 20]
test_win_rate = test_oversold["Target"].mean() * 100

# Baselines for context
train_baseline = train_df["Target"].mean() * 100
test_baseline = test_df["Target"].mean() * 100

print(f"Train Set (In-Sample):    {len(train_oversold):,} trades | Win Rate: {train_win_rate:.2f}% (Baseline: {train_baseline:.2f}%)")
print(f"Test Set (Out-of-Sample): {len(test_oversold):,} trades | Win Rate: {test_win_rate:.2f}% (Baseline: {test_baseline:.2f}%)")

Train Set (In-Sample):    733 trades | Win Rate: 55.93% (Baseline: 51.39%)
Test Set (Out-of-Sample): 268 trades | Win Rate: 55.22% (Baseline: 51.64%)


In [14]:
# Calculate raw price change in points/dollars
df["Price_Delta"] = df["Future_Close"] - df["close"]

# Check average gain during extreme oversold setups vs spread
oversold_df = df[df["RSI"] < 20]

avg_move = oversold_df["Price_Delta"].mean()
median_move = oversold_df["Price_Delta"].median()
avg_spread = oversold_df["spread"].mean()

print(f"Total Setups Analyzed: {len(oversold_df):,}")
print(f"Average Price Move:    ${avg_move:.3f}")
print(f"Median Price Move:     ${median_move:.3f}")

Total Setups Analyzed: 1,001
Average Price Move:    $0.516
Median Price Move:     $0.461


In [15]:
# 1. Filter for the 1,001 oversold setups
oversold_trades = df[df["RSI"] < 20].copy()

# 2. Group by Hour to inspect average dollar bounce and win rate across the day
hourly_edge = oversold_trades.groupby("Hour").agg(
    setups=("Target", "count"),
    win_rate=("Target", lambda x: f"{(x.mean() * 100):.1f}%"),
    avg_dollar_move=("Price_Delta", "mean")
).reset_index()

print(hourly_edge.sort_values(by="setups", ascending=False))

    Hour  setups win_rate  avg_dollar_move
13    13     114    51.8%         1.244912
14    14     106    45.3%        -0.188208
12    12      92    45.7%        -3.250076
5      5      77    48.1%        -1.589156
1      1      68    67.6%         2.143176
6      6      59    62.7%         0.886203
8      8      52    57.7%         1.358115
15    15      49    51.0%        -8.783245
11    11      43    51.2%         0.136395
0      0      42    71.4%        11.227429
22    23      36    66.7%         0.148083
21    22      31    61.3%         0.443645
19    19      31    35.5%        -1.828774
10    10      29    55.2%         2.033724
7      7      29    48.3%         3.213724
2      2      25    72.0%         6.450480
3      3      25    56.0%         1.115520
4      4      21    71.4%         0.569667
9      9      21    81.0%         1.744810
20    20      17    94.1%         2.894000
16    16      16    56.2%         4.158625
18    18      15    53.3%         1.833733
17    17   

In [ ]:
a